# Cane Corso Regression Experiment

В този notebook правя кратка тестова проба за **Linear Regression, Regularization and Testing**.

Използвам примерни Cane Corso данни, за да експериментирам с:

- Regression motivation
- Ordinary Least Squares
- Simulated / reference-style data
- RANSAC robust regression
- Polynomial regression
- Multi-dimensional regression
- Regularization
- Model testing

Идеята ми е да видя как различни regression модели могат да прогнозират тегло според възраст, храна и активност.

Това е учебна математическа проба, не ветеринарна препоръка.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

np.random.seed(42)

def mse(real, predicted):
    return np.mean((real - predicted) ** 2)

months = np.array([3, 4, 5, 6, 7, 8, 9, 10, 11, 12])
food_g = np.array([350, 450, 550, 650, 750, 850, 900, 950, 980, 1000])
activity_min = np.array([35, 40, 45, 50, 55, 60, 65, 65, 60, 55])

weight = 2.8 * months + 0.015 * food_g - 0.04 * activity_min + 4
weight = weight + np.random.normal(0, 1.5, size=len(months))

data = pd.DataFrame({
    "Месец": months,
    "Храна г/ден": food_g,
    "Активност мин/ден": activity_min,
    "Тегло кг": weight
})

data

In [ ]:
# Ordinary Least Squares: месец -> тегло

linear_a, linear_b = np.polyfit(months, weight, 1)
linear_pred = linear_a * months + linear_b

# Polynomial regression: месец -> тегло

poly_coeffs = np.polyfit(months, weight, 2)
poly_pred = np.polyval(poly_coeffs, months)

plt.scatter(months, weight, label="Примерни данни")
plt.plot(months, linear_pred, label="OLS Linear Regression")
plt.plot(months, poly_pred, label="Polynomial Regression")

plt.title("OLS и Polynomial Regression")
plt.xlabel("Месец")
plt.ylabel("Тегло кг")
plt.grid(True)
plt.legend()
plt.show()

print("OLS formula:")
print(f"тегло = {linear_a:.2f} * месец + {linear_b:.2f}")

print("OLS MSE:", mse(weight, linear_pred))
print("Polynomial MSE:", mse(weight, poly_pred))

In [ ]:
# RANSAC-style robust regression

weight_outlier = weight.copy()
weight_outlier[6] = 60  # outlier

def simple_ransac(x, y, iterations=300, threshold=4):
    best_a, best_b, best_inliers = None, None, None
    best_count = 0

    for _ in range(iterations):
        idx = np.random.choice(len(x), 2, replace=False)
        a, b = np.polyfit(x[idx], y[idx], 1)
        pred = a * x + b
        errors = np.abs(y - pred)
        inliers = errors < threshold

        if inliers.sum() > best_count:
            best_a, best_b = a, b
            best_inliers = inliers
            best_count = inliers.sum()

    return best_a, best_b, best_inliers

ransac_a, ransac_b, inliers = simple_ransac(months, weight_outlier)
ransac_pred = ransac_a * months + ransac_b

# Multi-dimensional regression: месец + храна + активност -> тегло

X = data[["Месец", "Храна г/ден", "Активност мин/ден"]].values
y = data["Тегло кг"].values

X_intercept = np.column_stack([X, np.ones(len(X))])
multi_coeffs, _, _, _ = np.linalg.lstsq(X_intercept, y, rcond=None)
multi_pred = X_intercept @ multi_coeffs

# Ridge regularization

X_mean = X.mean(axis=0)
X_std = X.std(axis=0)
X_scaled = (X - X_mean) / X_std
X_scaled_intercept = np.column_stack([X_scaled, np.ones(len(X_scaled))])

def ridge_regression(X, y, alpha):
    identity = np.eye(X.shape[1])
    identity[-1, -1] = 0
    return np.linalg.inv(X.T @ X + alpha * identity) @ X.T @ y

ridge_coeffs = ridge_regression(X_scaled_intercept, y, alpha=1)
ridge_pred = X_scaled_intercept @ ridge_coeffs

plt.scatter(months[inliers], weight_outlier[inliers], label="Нормални точки")
plt.scatter(months[~inliers], weight_outlier[~inliers], label="Outlier")
plt.plot(months, ransac_pred, label="RANSAC-style линия")

plt.title("RANSAC Robust Regression")
plt.xlabel("Месец")
plt.ylabel("Тегло кг")
plt.grid(True)
plt.legend()
plt.show()

print("Multi-dimensional formula:")
print(f"тегло = {multi_coeffs[0]:.2f} * месец + {multi_coeffs[1]:.4f} * храна + {multi_coeffs[2]:.4f} * активност + {multi_coeffs[3]:.2f}")

In [ ]:
results = pd.DataFrame({
    "Модел": [
        "OLS Linear Regression",
        "Polynomial Regression",
        "RANSAC-style Regression",
        "Multi-dimensional Regression",
        "Ridge Regularization"
    ],
    "MSE": [
        mse(weight, linear_pred),
        mse(weight, poly_pred),
        mse(weight_outlier[inliers], ransac_pred[inliers]),
        mse(y, multi_pred),
        mse(y, ridge_pred)
    ]
})

display(results.sort_values("MSE"))

plt.barh(results["Модел"], results["MSE"])
plt.title("Model Testing — MSE сравнение")
plt.xlabel("MSE")
plt.grid(axis="x")
plt.show()

assert linear_a > 0
assert ransac_a > 0
assert len(results) == 5
assert results["MSE"].isna().sum() == 0
assert (results["MSE"] >= 0).all()

print("Tests passed")